In [1]:
import pandas as pd
import numpy as np

data = {
    "OrderID": [501, 502, 503, 504, 505, 503, 507, 508, 509, 510],
    "Product": [
        "Laptop", "Phone", "Tablet", "Laptop", "Phone",
        "Tablet", "Laptop", "Phone", "Tablet", "Laptop"
    ],
    "Region": [
        "North", "South", "East", "West", "South",
        "East", "North", "south", "West", "North"
    ],
    "UnitsSold": [5, 12, 8, 6, 15, 8, 7, 11, 9, 200],
    "UnitPrice": [1200, 800, 400, 1200, 800, 400, 1200, 800, 400, 1200],
    "OrderDate": [
        "2026-01-05", "2026-01-07", "2026-01-09", "2026-01-11",
        "2026-01-13", "2026-01-09", "2026-01-15", "2026-01-17",
        "2026-01-19", "2026-01-21"
    ]
}

df = pd.DataFrame(data)
df

,OrderID,Product,Region,UnitsSold,UnitPrice,OrderDate
0,501,Laptop,North,5,1200,2026-01-05
1,502,Phone,South,12,800,2026-01-07
2,503,Tablet,East,8,400,2026-01-09
3,504,Laptop,West,6,1200,2026-01-11
4,505,Phone,South,15,800,2026-01-13
5,503,Tablet,East,8,400,2026-01-09
6,507,Laptop,North,7,1200,2026-01-15
7,508,Phone,south,11,800,2026-01-17
8,509,Tablet,West,9,400,2026-01-19
9,510,Laptop,North,200,1200,2026-01-21


In [9]:
#Inspect the dataset using shape, info() and describe().
df.shape
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   OrderID    10 non-null     int64
 1   Product    10 non-null     str  
 2   Region     10 non-null     str  
 3   UnitsSold  10 non-null     int64
 4   UnitPrice  10 non-null     int64
 5   OrderDate  10 non-null     str  
dtypes: int64(3), str(3)
memory usage: 612.0 bytes


,OrderID,UnitsSold,UnitPrice
count,10.000000,10.000000,10.000000
mean,505.200000,28.100000,840.000000
std,3.119829,60.473043,350.238014
min,501.000000,5.000000,400.000000
25%,503.000000,7.250000,500.000000
50%,504.500000,8.500000,800.000000
75%,507.750000,11.750000,1200.000000
max,510.000000,200.000000,1200.000000


In [ ]:
#Find the number of duplicated rows.
df.duplicated()
#df.duplicated().sum()

0    False
1    False
2    False
3    False
4    False
5     True
6    False
7    False
8    False
9    False
dtype: bool

In [12]:
#Display rows with duplicate OrderID values.
df[df.duplicated(keep=False)]

,OrderID,Product,Region,UnitsSold,UnitPrice,OrderDate
2,503,Tablet,East,8,400,2026-01-09
5,503,Tablet,East,8,400,2026-01-09


In [13]:
#Remove exact duplicate rows
df = df.drop_duplicates()

In [14]:
#Standardize Region values
df["Region"] = df["Region"].str.strip().str.title()

df["Region"].value_counts()

Region
North    3
South    3
West     2
East     1
Name: count, dtype: int64

In [15]:
#Convert OrderDate to datetime

df["OrderDate"] = pd.to_datetime(df["OrderDate"])

df.dtypes

OrderID               int64
Product                 str
Region                  str
UnitsSold             int64
UnitPrice             int64
OrderDate    datetime64[us]
dtype: object

In [16]:
#Create the Revenue column
df["Revenue"] = df["UnitsSold"] * df["UnitPrice"]

df[["OrderID", "UnitsSold", "UnitPrice", "Revenue"]]

,OrderID,UnitsSold,UnitPrice,Revenue
0,501,5,1200,6000
1,502,12,800,9600
2,503,8,400,3200
3,504,6,1200,7200
4,505,15,800,12000
6,507,7,1200,8400
7,508,11,800,8800
8,509,9,400,3600
9,510,200,1200,240000


In [ ]:
#Identify outliers using the IQR method
Q1 = df["UnitsSold"].quantile(0.25)
Q3 = df["UnitsSold"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

#Any UnitsSold value below -0.5 or above 19.5 is flagged. Here, 200 units is the outlier.

Q1: 7.0
Q3: 12.0
IQR: 5.0
Lower bound: -0.5
Upper bound: 19.5


In [19]:
#Create the IsOutlier column
df["IsOutlier"] = (
    (df["UnitsSold"] < lower_bound) |
    (df["UnitsSold"] > upper_bound)
)

df.loc[df["IsOutlier"], ["OrderID", "UnitsSold", "IsOutlier"]]

,OrderID,UnitsSold,IsOutlier
9,510,200,True


In [ ]:
#Create the cleaned dataset
cleaned_df = df[~df["IsOutlier"]].copy()

#The ~ operator reverses the Boolean condition, keeping rows where IsOutlier is False.

In [21]:
#Reset the index and verify
cleaned_df = cleaned_df.reset_index(drop=True)

print("Final shape:", cleaned_df.shape)
cleaned_df

Final shape: (8, 8)


,OrderID,Product,Region,UnitsSold,UnitPrice,OrderDate,Revenue,IsOutlier
0,501,Laptop,North,5,1200,2026-01-05,6000,False
1,502,Phone,South,12,800,2026-01-07,9600,False
2,503,Tablet,East,8,400,2026-01-09,3200,False
3,504,Laptop,West,6,1200,2026-01-11,7200,False
4,505,Phone,South,15,800,2026-01-13,12000,False
5,507,Laptop,North,7,1200,2026-01-15,8400,False
6,508,Phone,South,11,800,2026-01-17,8800,False
7,509,Tablet,West,9,400,2026-01-19,3600,False


This exercise removes the outlier for analysis, but in a real project, an unusually large sale could be legitimate. 
Investigate it before deleting it. Data cleaning should not quietly erase reality just because reality looks unusual.